In [17]:
import sys

import pandas as pd

from from_root import from_root

sys.path.insert(0, str(from_root("src")))

from read_and_write_docs import read_rds

In [60]:
df = read_rds("/Users/user/Downloads/raw_60_phrase_occurrence_score_raw.rds")

In [80]:
df.shape

(129558, 17)

In [65]:
def llr_unknown_vs_nocontext(df):
    """
    positive → unknown-context score is more probable
    negative → no-context score is more probable
    zero → equally probable
    """
    
    df['llr_unknown_vs_nocontext'] = df['unknown_sum_log_probs'] - df['no_context_sum_log_probs']
    
    return df

In [66]:
def create_per_phrase_table(
    per_occurrence_table: pd.DataFrame,
    include_sums: bool = False
) -> pd.DataFrame:
    """Create an aggregated version of the table containing one row per phrase.

    By default, numeric non-key columns are averaged over occurrences.
    If include_sums=True, summed versions of those same numeric columns are also added
    with a '_sum' suffix.
    """

    key_cols = [
        "data_type", "corpus", "scoring_model", "max_context_tokens", "problem",
        "known_author", "unknown_author", "target", "completed",
        "phrase_num", "phrase", "tokens", "num_tokens"
    ]
    skip_cols = ["phrase_occurrence"]

    # Group keys must exclude list-typed 'tokens'
    group_keys = [
        "data_type", "corpus", "scoring_model", "max_context_tokens", "problem",
        "known_author", "unknown_author", "target", "completed",
        "phrase_num", "phrase", "num_tokens"
    ]

    # Columns to aggregate, preserving original order
    agg_cols = [
        c for c in per_occurrence_table.columns
        if c not in set(key_cols + skip_cols)
    ]

    # Only numeric columns can be averaged/summed here
    numeric_agg_cols = [
        c for c in agg_cols
        if pd.api.types.is_numeric_dtype(per_occurrence_table[c])
    ]

    # Build aggregation spec
    agg_spec = {c: "mean" for c in numeric_agg_cols}

    if include_sums:
        for c in numeric_agg_cols:
            agg_spec[f"{c}_sum"] = (c, "sum")

        per_phrase_table = (
            per_occurrence_table
            .groupby(group_keys, as_index=False)
            .agg(**{
                **{c: (c, "mean") for c in numeric_agg_cols},
                **{f"{c}_sum": (c, "sum") for c in numeric_agg_cols},
            })
        )
    else:
        per_phrase_table = (
            per_occurrence_table
            .groupby(group_keys, as_index=False)[numeric_agg_cols]
            .mean()
        )

    # Bring tokens back
    tokens_map = (
        per_occurrence_table[
            [
                "data_type", "corpus", "scoring_model", "max_context_tokens",
                "problem", "known_author", "unknown_author", "target", "completed",
                "phrase_num", "phrase", "tokens"
            ]
        ]
        .drop_duplicates(
            subset=[
                "data_type", "corpus", "scoring_model", "max_context_tokens",
                "problem", "known_author", "unknown_author", "target", "completed",
                "phrase_num", "phrase"
            ]
        )
    )

    per_phrase_table = per_phrase_table.merge(
        tokens_map,
        on=[
            "data_type", "corpus", "scoring_model", "max_context_tokens",
            "problem", "known_author", "unknown_author", "target", "completed",
            "phrase_num", "phrase"
        ],
        how="left"
    )

    # Reorder columns
    ordered_cols = key_cols + numeric_agg_cols
    if include_sums:
        ordered_cols += [f"{c}_sum" for c in numeric_agg_cols]

    per_phrase_table = per_phrase_table[ordered_cols]

    return per_phrase_table

In [75]:
import pandas as pd


def create_summary_by_token_num(per_phrase_table: pd.DataFrame) -> pd.DataFrame:
    """
    For each problem/metadata group and each token threshold t (2, 3, ...),
    compute column-wise sums over rows where num_tokens >= t.

    Returns one row per problem per threshold.
    """

    key_cols = [
        'data_type', 'corpus', 'scoring_model', 'max_context_tokens', 'problem',
        'known_author', 'unknown_author', 'target', 'completed'
    ]
    skip_cols = ['phrase_num', 'phrase', 'tokens', 'num_tokens']

    if 'num_tokens' not in per_phrase_table.columns:
        raise ValueError("per_phrase_table must contain a 'num_tokens' column.")

    sum_cols = [
        c for c in per_phrase_table.columns
        if c not in set(skip_cols + key_cols)
        and pd.api.types.is_numeric_dtype(per_phrase_table[c])
    ]

    rows = []

    for group_values, group_df in per_phrase_table.groupby(key_cols, dropna=False):
        token_thresholds = sorted(group_df['num_tokens'].dropna().unique())

        for t in token_thresholds:
            filt = group_df[group_df['num_tokens'] >= t]
            sums = filt[sum_cols].sum(numeric_only=True)

            row = dict(zip(key_cols, group_values))
            row['min_token_size'] = int(t)
            row['n_rows'] = int(len(filt))
            row.update(sums.to_dict())
            rows.append(row)

    out = pd.DataFrame(rows).sort_values(key_cols + ['min_token_size']).reset_index(drop=True)

    ordered_cols = (
        key_cols
        + ['min_token_size', 'n_rows']
        + [c for c in sum_cols if c in out.columns]
    )

    return out[ordered_cols]

In [69]:
df_with_scores = llr_unknown_vs_nocontext(df)

In [70]:
phrase_scores = create_per_phrase_table(
    per_occurrence_table=df_with_scores,
    include_sums=True
)

In [76]:
score_by_tokens = create_summary_by_token_num(phrase_scores)

In [81]:
score_by_tokens.shape

(1321, 17)

In [79]:
score_by_tokens.head(20)

,data_type,corpus,scoring_model,max_context_tokens,problem,known_author,unknown_author,target,completed,min_token_size,n_rows,no_context_sum_log_probs,unknown_sum_log_probs,llr_unknown_vs_nocontext,no_context_sum_log_probs_sum,unknown_sum_log_probs_sum,llr_unknown_vs_nocontext_sum
0,test,ACL,gpt2,60.0,"Ibekwe-SanJuan, Fidelia vs Ibekwe-SanJuan, Fid...","Ibekwe-SanJuan, Fidelia","Ibekwe-SanJuan, Fidelia",True,True,1,231,-3412.582874,-1594.966970,1817.615904,-6433.882143,-2906.216016,3527.666127
1,test,ACL,gpt2,60.0,"Ibekwe-SanJuan, Fidelia vs Ibekwe-SanJuan, Fid...","Ibekwe-SanJuan, Fidelia","Ibekwe-SanJuan, Fidelia",True,True,2,127,-2015.050190,-970.775691,1044.274499,-3555.959951,-1666.116765,1889.843186
2,test,ACL,gpt2,60.0,"Ibekwe-SanJuan, Fidelia vs Ibekwe-SanJuan, Fid...","Ibekwe-SanJuan, Fidelia","Ibekwe-SanJuan, Fidelia",True,True,3,28,-600.696632,-328.757965,271.938667,-820.805244,-454.684981,366.120263
3,test,ACL,gpt2,60.0,"Ibekwe-SanJuan, Fidelia vs Ibekwe-SanJuan, Fid...","Ibekwe-SanJuan, Fidelia","Ibekwe-SanJuan, Fidelia",True,True,4,4,-123.853603,-97.411926,26.441677,-167.534603,-136.339875,31.194728
4,test,ACL,gpt2,60.0,"Ibekwe-SanJuan, Fidelia vs Ibekwe-SanJuan, Fid...","Ibekwe-SanJuan, Fidelia","Ibekwe-SanJuan, Fidelia",True,True,5,1,-43.681000,-38.927950,4.753051,-87.362000,-77.855899,9.506101
5,test,ACL,gpt2,60.0,"Ibekwe-SanJuan, Fidelia vs Ide, Nancy","Ibekwe-SanJuan, Fidelia","Ide, Nancy",False,True,1,143,-1987.183682,-826.836939,1160.346743,-3328.825026,-1287.326629,2041.498397
6,test,ACL,gpt2,60.0,"Ibekwe-SanJuan, Fidelia vs Ide, Nancy","Ibekwe-SanJuan, Fidelia","Ide, Nancy",False,True,2,76,-1163.884179,-505.390415,658.493764,-1716.800604,-696.206632,1020.593972
7,test,ACL,gpt2,60.0,"Ibekwe-SanJuan, Fidelia vs Ide, Nancy","Ibekwe-SanJuan, Fidelia","Ide, Nancy",False,True,3,19,-329.878894,-163.091323,166.787570,-329.878894,-163.091323,166.787570
8,test,ACL,gpt2,60.0,"Ide, Nancy vs Ide, Nancy","Ide, Nancy","Ide, Nancy",True,True,1,144,-1881.278898,-743.493346,1137.785553,-3543.507009,-1316.240884,2227.266125
9,test,ACL,gpt2,60.0,"Ide, Nancy vs Ide, Nancy","Ide, Nancy","Ide, Nancy",True,True,2,64,-868.319311,-368.347711,499.971600,-1596.674337,-654.893918,941.780419
